# 第19章（四）：递归
# Chapter 19 (Part 4): Recursion

**Cambridge A-Level Computer Science (9618)**

---

## 本节学习目标 (Learning Objectives)

1. 理解递归的本质：**基线条件 (Base Case)** 和 **递归条件 (Recursive Case)**
2. 理解**调用栈 (Call Stack)** 和**栈帧 (Stack Frame)** 的工作原理
3. 掌握经典递归算法：阶乘、斐波那契、汉诺塔
4. 比较**递归 vs 迭代**，理解何时使用递归
5. 了解**尾递归优化**和**记忆化 (Memoization)**

## 1. 递归的本质

### 1.1 什么是递归？

> **递归 (Recursion)** 是指一个函数直接或间接地调用自身。

### 1.2 为什么递归有效？

递归的核心思想是：**将一个大问题分解为更小的同类问题**。

每个递归函数必须有两个部分：

1. **基线条件 (Base Case)**：最简单的情况，可以直接返回结果，**不再递归调用**
2. **递归条件 (Recursive Case)**：将问题缩小，调用自身来解决更小的子问题

### 1.3 为什么基线条件至关重要？

没有基线条件 → 无限递归 → **栈溢出 (Stack Overflow)**！

```python
# 危险！没有基线条件的递归
def infinite_recursion(n):
    return infinite_recursion(n + 1)  # 永远不会停止！
```

Python 默认递归深度限制是 **1000 层**，超过会报 `RecursionError`。

In [ ]:
# ============================================
# 递归的第一个例子：阶乘 (Factorial)
# ============================================

def factorial_recursive(n, depth=0):
    '''
    递归计算 n! = n * (n-1) * (n-2) * ... * 1
    基线条件: 0! = 1 或 1! = 1
    递归条件: n! = n * (n-1)!
    '''
    indent = '  ' * depth
    print(f'{indent}调用 factorial({n})')
    
    # 基线条件 (Base Case)
    if n <= 1:
        print(f'{indent}-> 基线条件: factorial({n}) = 1')
        return 1
    
    # 递归条件 (Recursive Case)
    result = n * factorial_recursive(n - 1, depth + 1)
    print(f'{indent}<- 返回: factorial({n}) = {n} * factorial({n-1}) = {result}')
    return result

print('=== 阶乘递归过程 ===')
print(f'\n计算 5! = ?\n')
result = factorial_recursive(5)
print(f'\n最终结果: 5! = {result}')


## 2. 调用栈 (Call Stack) 与栈帧 (Stack Frame)

### 2.1 调用栈的工作原理

每次函数调用时，系统会在**调用栈 (Call Stack)** 上创建一个**栈帧 (Stack Frame)**：

- 栈帧包含：局部变量、参数、返回地址
- 函数返回时，栈帧被弹出
- 递归调用会创建多个栈帧

### 2.2 为什么要理解调用栈？

1. **调试**：理解递归的执行顺序
2. **空间复杂度**：递归的空间复杂度 = O(递归深度)
3. **栈溢出**：递归太深会耗尽栈空间

In [ ]:
# ============================================
# 可视化：调用栈的展开与回溯
# ============================================
import matplotlib.pyplot as plt
import matplotlib.patches as patches

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# 左图：展开阶段（入栈）
ax1 = axes[0]
ax1.set_xlim(0, 10)
ax1.set_ylim(-0.5, 12)
ax1.axis('off')
ax1.set_title('展开阶段 (Unwinding) - 入栈', fontsize=14, fontweight='bold')

frames_expand = [
    ('factorial(5)', '5 * factorial(4) = ?', '#e74c3c'),
    ('factorial(4)', '4 * factorial(3) = ?', '#e67e22'),
    ('factorial(3)', '3 * factorial(2) = ?', '#f1c40f'),
    ('factorial(2)', '2 * factorial(1) = ?', '#2ecc71'),
    ('factorial(1)', 'Base Case! return 1', '#3498db'),
]

for i, (name, detail, color) in enumerate(frames_expand):
    y = 10 - i * 2
    rect = patches.FancyBboxPatch((1, y), 8, 1.5,
        boxstyle='round,pad=0.1', facecolor=color, alpha=0.3,
        edgecolor=color, linewidth=2)
    ax1.add_patch(rect)
    ax1.text(5, y + 0.75, name, ha='center', va='center',
             fontsize=12, fontweight='bold')
    ax1.text(5, y + 0.75, detail, ha='center', va='center',
             fontsize=10, color='gray')
    if i > 0:
        ax1.annotate('', xy=(5, y + 1.5), xytext=(5, y + 2.2),
                    arrowprops=dict(arrowstyle='->', color=color, lw=2))

ax1.text(5, -0.3, '\u2193 栈增长方向', ha='center', va='center', fontsize=11, color='gray')

# 右图：回溯阶段（出栈）
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(-0.5, 12)
ax2.axis('off')
ax2.set_title('回溯阶段 (Backtracking) - 出栈', fontsize=14, fontweight='bold')

frames_return = [
    ('factorial(5)', '5 * 24 = 120 \u2190 最终结果', '#e74c3c'),
    ('factorial(4)', '4 * 6 = 24', '#e67e22'),
    ('factorial(3)', '3 * 2 = 6', '#f1c40f'),
    ('factorial(2)', '2 * 1 = 2', '#2ecc71'),
    ('factorial(1)', 'return 1 \u2190 基线条件', '#3498db'),
]

for i, (name, detail, color) in enumerate(frames_return):
    y = 10 - i * 2
    rect = patches.FancyBboxPatch((1, y), 8, 1.5,
        boxstyle='round,pad=0.1', facecolor=color, alpha=0.3,
        edgecolor=color, linewidth=2)
    ax2.add_patch(rect)
    ax2.text(5, y + 0.75, name, ha='center', va='center',
             fontsize=12, fontweight='bold')
    ax2.text(5, y + 0.75, detail, ha='center', va='center',
             fontsize=10, color='gray')
    if i > 0:
        ax2.annotate('', xy=(5, y + 2.2), xytext=(5, y + 1.5),
                    arrowprops=dict(arrowstyle='->', color=color, lw=2))

ax2.text(5, -0.3, '\u2191 返回方向', ha='center', va='center', fontsize=11, color='gray')

plt.tight_layout()
plt.show()
print('左图：每次递归调用会创建新的栈帧，压入调用栈')


## 3. 经典递归算法

### 3.1 斐波那契数列 (Fibonacci Sequence)

定义：F(0) = 0, F(1) = 1, F(n) = F(n-1) + F(n-2)

序列：0, 1, 1, 2, 3, 5, 8, 13, 21, 34, ...

In [ ]:
# ============================================
# 斐波那契数列：朴素递归 vs 优化版本
# ============================================
import time

# 方法1: 朴素递归 - O(2^n) 指数级！
call_count = 0
def fib_naive(n):
    '''朴素递归：大量重复计算'''
    global call_count
    call_count += 1
    if n <= 1:
        return n
    return fib_naive(n - 1) + fib_naive(n - 2)

# 方法2: 记忆化递归 - O(n)
def fib_memo(n, memo=None):
    '''记忆化递归：缓存已计算的结果'''
    if memo is None:
        memo = {}
    if n in memo:
        return memo[n]
    if n <= 1:
        return n
    memo[n] = fib_memo(n - 1, memo) + fib_memo(n - 2, memo)
    return memo[n]

# 方法3: 迭代 - O(n)时间, O(1)空间
def fib_iterative(n):
    '''迭代版本：最高效'''
    if n <= 1:
        return n
    a, b = 0, 1
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b

# 对比
print('=== 斐波那契数列：三种方法对比 ===')
print(f'\n{"n":>4} | {"朴素递归":>12} | {"调用次数":>10} | {"记忆化":>12} | {"迭代":>12}')
print('-' * 65)

for n in [5, 10, 15, 20, 25, 30]:
    # 朴素递归
    call_count = 0
    start = time.perf_counter()
    r1 = fib_naive(n)
    t1 = (time.perf_counter() - start) * 1000
    calls = call_count
    
    # 记忆化
    start = time.perf_counter()
    r2 = fib_memo(n)
    t2 = (time.perf_counter() - start) * 1000
    
    # 迭代
    start = time.perf_counter()
    r3 = fib_iterative(n)
    t3 = (time.perf_counter() - start) * 1000
    
    print(f'{n:>4} | {t1:>10.3f}ms | {calls:>10,} | {t2:>10.3f}ms | {t3:>10.3f}ms')

print('\n观察：朴素递归的调用次数爆炸式增长！')


In [ ]:
# ============================================
# 可视化：斐波那契递归树 -- 展示重复计算
# ============================================
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 16)
ax.set_ylim(-1, 9)
ax.axis('off')
ax.set_title('斐波那契递归树 fib(5) - 红色节点表示重复计算',
             fontsize=14, fontweight='bold')

# 手动绘制fib(5)的递归树
nodes = [
    # (x, y, label, color, parent_x, parent_y)
    (8, 8, 'fib(5)', '#3498db', None, None),
    (5, 6.5, 'fib(4)', '#3498db', 8, 8),
    (11, 6.5, 'fib(3)', '#e74c3c', 8, 8),
    (3.5, 5, 'fib(3)', '#e74c3c', 5, 6.5),
    (6.5, 5, 'fib(2)', '#e74c3c', 5, 6.5),
    (9.5, 5, 'fib(2)', '#e74c3c', 11, 6.5),
    (12.5, 5, 'fib(1)', '#2ecc71', 11, 6.5),
    (2.5, 3.5, 'fib(2)', '#e74c3c', 3.5, 5),
    (4.5, 3.5, 'fib(1)', '#2ecc71', 3.5, 5),
    (5.8, 3.5, 'fib(1)', '#2ecc71', 6.5, 5),
    (7.2, 3.5, 'fib(0)', '#2ecc71', 6.5, 5),
    (8.8, 3.5, 'fib(1)', '#2ecc71', 9.5, 5),
    (10.2, 3.5, 'fib(0)', '#2ecc71', 9.5, 5),
    (1.8, 2, 'fib(1)', '#2ecc71', 2.5, 3.5),
    (3.2, 2, 'fib(0)', '#2ecc71', 2.5, 3.5),
]

for x, y, label, color, px, py in nodes:
    # 画连线
    if px is not None:
        ax.plot([px, x], [py - 0.3, y + 0.3], 'gray', linewidth=1, alpha=0.5)
    # 画节点
    circle = plt.Circle((x, y), 0.45, color=color, alpha=0.3, ec=color, linewidth=1.5)
    ax.add_patch(circle)
    ax.text(x, y, label, ha='center', va='center', fontsize=8, fontweight='bold')

# 图例
ax.text(0.5, 0.5, '蓝色: 唯一计算', fontsize=11, color='#3498db',
        bbox=dict(facecolor='white', alpha=0.8, edgecolor='#3498db'))
ax.text(4, 0.5, '红色: 重复计算!', fontsize=11, color='#e74c3c',
        bbox=dict(facecolor='white', alpha=0.8, edgecolor='#e74c3c'))
ax.text(8, 0.5, '绿色: 基线条件', fontsize=11, color='#2ecc71',
        bbox=dict(facecolor='white', alpha=0.8, edgecolor='#2ecc71'))

ax.text(8, -0.5, 'fib(5)总共需要15次调用，其中大量是重复的！',
        ha='center', fontsize=12, color='#7f8c8d')

plt.tight_layout()


### 3.2 汉诺塔 (Tower of Hanoi)

**问题描述**：
- 有三根柱子 A、B、C
- A 上有 n 个大小不同的圆盘，从上到下由小到大排列
- 目标：将所有圆盘从 A 移到 C
- 规则：每次只能移动一个圆盘，且大盘不能放在小盘上面

**递归思路**：
1. 将上面 n-1 个圆盘从 A 移到 B（借助 C）
2. 将最大的圆盘从 A 移到 C
3. 将 n-1 个圆盘从 B 移到 C（借助 A）

In [ ]:
# ============================================
# 汉诺塔递归实现
# ============================================

move_count = 0

def hanoi(n, source='A', target='C', auxiliary='B', depth=0):
    '''
    汉诺塔递归解法
    n: 圆盘数量
    source: 源柱子
    target: 目标柱子
    auxiliary: 辅助柱子
    '''
    global move_count
    indent = '  ' * depth
    
    if n == 1:
        # 基线条件：只有一个圆盘，直接移动
        move_count += 1
        print(f'{indent}移动 圆盘1: {source} -> {target}  (第{move_count}步)')
        return
    
    # 第1步：将n-1个圆盘从source移到auxiliary
    if depth < 2:
        print(f'{indent}[将{n-1}个圆盘从{source}移到{auxiliary}]')
    hanoi(n - 1, source, auxiliary, target, depth + 1)
    
    # 第2步：将最大圆盘从source移到target
    move_count += 1
    print(f'{indent}移动 圆盘{n}: {source} -> {target}  (第{move_count}步)')
    
    # 第3步：将n-1个圆盘从auxiliary移到target
    if depth < 2:
        print(f'{indent}[将{n-1}个圆盘从{auxiliary}移到{target}]')
    hanoi(n - 1, auxiliary, target, source, depth + 1)

# 演示
for n in [3, 4]:
    move_count = 0
    print(f'\n{"=" * 50}')
    print(f'汉诺塔: {n}个圆盘')
    print(f'{"=" * 50}')
    hanoi(n)


In [ ]:
# ============================================
# 可视化：汉诺塔步骤演示
# ============================================
import matplotlib.pyplot as plt
import matplotlib.patches as patches

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

def draw_hanoi_state(pegs, step_num, move_desc, ax):
    '''画出汉诺塔的某个状态'''
    ax.set_xlim(0, 12)
    ax.set_ylim(0, 5)
    ax.axis('off')
    ax.set_title(f'步骤 {step_num}: {move_desc}', fontsize=10)
    
    colors = ['#e74c3c', '#3498db', '#2ecc71', '#f1c40f']
    peg_x = [2, 6, 10]
    peg_names = ['A', 'B', 'C']
    
    for i, (px, name) in enumerate(zip(peg_x, peg_names)):
        # 画柱子
        ax.plot([px, px], [0.5, 4], color='#7f8c8d', linewidth=3)
        ax.plot([px-1.5, px+1.5], [0.5, 0.5], color='#7f8c8d', linewidth=3)
        ax.text(px, 0.1, name, ha='center', va='center', fontsize=10, fontweight='bold')
        
        # 画圆盘
        for j, disk in enumerate(pegs[i]):
            width = 0.5 + disk * 0.6
            rect = patches.FancyBboxPatch(
                (px - width/2, 0.5 + j * 0.5), width, 0.4,
                boxstyle='round,pad=0.05',
                facecolor=colors[disk-1], alpha=0.8,
                edgecolor='white', linewidth=1)
            ax.add_patch(rect)
            ax.text(px, 0.7 + j * 0.5, str(disk), ha='center', va='center',
                    fontsize=9, color='white', fontweight='bold')

# 模拟3个圆盘的汉诺塔
states = [
    ([[3,2,1], [], []], 0, '初始状态'),
    ([[3,2], [], [1]], 1, '圆盘1: A->C'),
    ([[3], [2], [1]], 2, '圆盘2: A->B'),
    ([[3], [2,1], []], 3, '圆盘1: C->B'),
    ([[], [2,1], [3]], 4, '圆盘3: A->C'),
    ([[1], [2], [3]], 5, '圆盘1: B->A'),
    ([[1], [], [3,2]], 6, '圆盘2: B->C'),
    ([[], [], [3,2,1]], 7, '圆盘1: A->C  完成!'),
]

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for idx, (pegs, step, desc) in enumerate(states):
    ax = axes[idx // 4][idx % 4]
    draw_hanoi_state(pegs, step, desc, ax)

plt.suptitle('汉诺塔演示: 3个圆盘的完整解决过程 (7步)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()


## 4. 递归 vs 迭代

### 4.1 什么时候用递归？

| 场景 | 推荐 | 原因 |
|------|------|------|
| 树的遍历 | 递归 | 树的结构天然适合递归 |
| 分治算法（归并排序、快排） | 递归 | 分解-解决-合并 |
| 简单的循环计算 | 迭代 | 更高效，无栈开销 |
| 深度很大的问题 | 迭代 | 避免栈溢出 |
| 有重复子问题 | 记忆化递归或动态规划 | 避免重复计算 |

### 4.2 递归的优缺点

| 优点 | 缺点 |
|------|------|
| 代码简洁优雅 | 栈空间开销 |
| 适合分治问题 | 可能有重复计算 |
| 自然表达递归结构 | 递归太深会栈溢出 |
| 容易理解和证明正确性 | 通常比迭代慢（函数调用开销） |

In [ ]:
# ============================================
# 递归 vs 迭代的性能对比
# ============================================
import time
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 阶乘：递归 vs 迭代
def fact_recursive(n):
    if n <= 1: return 1
    return n * fact_recursive(n - 1)

def fact_iterative(n):
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result

sizes = [10, 50, 100, 200, 500, 800]
rec_times = []
iter_times = []

for n in sizes:
    # 递归
    start = time.perf_counter()
    for _ in range(1000):
        fact_recursive(n)
    rec_times.append((time.perf_counter() - start) * 1000 / 1000)
    
    # 迭代
    start = time.perf_counter()
    for _ in range(1000):
        fact_iterative(n)
    iter_times.append((time.perf_counter() - start) * 1000 / 1000)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(sizes, rec_times, 'ro-', linewidth=2, markersize=8, label='递归 (Recursive)')
ax.plot(sizes, iter_times, 'bs-', linewidth=2, markersize=8, label='迭代 (Iterative)')
ax.set_xlabel('n', fontsize=12)
ax.set_ylabel('平均运行时间 (ms)', fontsize=12)
ax.set_title('阶乘计算：递归 vs 迭代', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('观察：迭代版本通常更快，因为没有函数调用的开销')


## 5. 递归在树遍历、分治算法中的应用

递归在以下场景中是最自然的选择：

In [ ]:
# ============================================
# 递归应用：目录树遍历（模拟文件系统）
# ============================================

def build_file_tree():
    '''模拟一个文件系统目录结构'''
    return {
        'name': 'root',
        'type': 'dir',
        'children': [
            {'name': 'documents', 'type': 'dir', 'children': [
                {'name': 'essay.docx', 'type': 'file', 'size': 25},
                {'name': 'notes.txt', 'type': 'file', 'size': 5},
                {'name': 'photos', 'type': 'dir', 'children': [
                    {'name': 'vacation.jpg', 'type': 'file', 'size': 3500},
                    {'name': 'family.png', 'type': 'file', 'size': 2800},
                ]}
            ]},
            {'name': 'music', 'type': 'dir', 'children': [
                {'name': 'song1.mp3', 'type': 'file', 'size': 4000},
                {'name': 'song2.mp3', 'type': 'file', 'size': 3500},
            ]},
            {'name': 'readme.txt', 'type': 'file', 'size': 2},
        ]
    }

def print_tree(node, indent=0):
    '''递归打印目录树'''
    prefix = '  ' * indent + ('|- ' if indent > 0 else '')
    if node['type'] == 'dir':
        print(f'{prefix}[DIR] {node["name"]}/')
        for child in node.get('children', []):
            print_tree(child, indent + 1)
    else:
        print(f'{prefix}[FILE] {node["name"]} ({node["size"]}KB)')

def total_size(node):
    '''递归计算目录总大小'''
    if node['type'] == 'file':
        return node['size']
    return sum(total_size(child) for child in node.get('children', []))

tree = build_file_tree()
print('=== 目录树遍历（递归）===')
print_tree(tree)
print(f'\n总大小: {total_size(tree)} KB')
print('\n这就是为什么递归天然适合树结构！')


## 6. 深入：尾递归优化与记忆化

### 6.1 尾递归 (Tail Recursion)

**尾递归**是指递归调用是函数的最后一个操作。

```python
# 普通递归：递归调用后还要乘以n
def factorial(n):
    if n <= 1: return 1
    return n * factorial(n - 1)  # 需要等递归返回后再计算

# 尾递归：递归调用是最后的操作
def factorial_tail(n, accumulator=1):
    if n <= 1: return accumulator
    return factorial_tail(n - 1, n * accumulator)  # 直接返回递归调用
```

**为什么尾递归好？** 理论上编译器可以优化尾递归，使其不需要额外的栈帧（复用当前帧）。

> **注意**：Python **不支持** 尾递归优化！但很多其他语言（Scheme, Haskell, Scala）支持。

### 6.2 记忆化 (Memoization)

记忆化是一种优化技术：**缓存已经计算过的结果，避免重复计算**。

Python 提供了内置的装饰器 `@functools.lru_cache`：

In [ ]:
# ============================================
# 记忆化 (Memoization) 演示
# ============================================
from functools import lru_cache
import time

# 使用 Python 内置的记忆化装饰器
@lru_cache(maxsize=None)
def fib_cached(n):
    '''带缓存的斐波那契'''
    if n <= 1:
        return n
    return fib_cached(n - 1) + fib_cached(n - 2)

# 对比
print('=== 记忆化的威力 ===')
print(f'\n{"n":>6} | {"朴素递归":>12} | {"记忆化递归":>12} | {"加速比":>8}')
print('-' * 50)

for n in [10, 20, 25, 30, 35]:
    # 朴素递归
    start = time.perf_counter()
    r1 = fib_naive(n)
    t1 = (time.perf_counter() - start) * 1000
    
    # 记忆化
    fib_cached.cache_clear()
    start = time.perf_counter()
    r2 = fib_cached(n)
    t2 = (time.perf_counter() - start) * 1000
    
    ratio = t1 / t2 if t2 > 0 else float('inf')
    print(f'{n:>6} | {t1:>10.3f}ms | {t2:>10.3f}ms | {ratio:>7.0f}x')

print('\n记忆化将O(2^n)降低到O(n)，效果惊人！')


In [ ]:
# ============================================
# 可视化：朴素递归 vs 记忆化的调用次数
# ============================================
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 计算朴素递归的调用次数
def count_calls_naive(n):
    if n <= 1: return 1
    return 1 + count_calls_naive(n-1) + count_calls_naive(n-2)

# 记忆化只需要 2n-1 次调用
ns = list(range(1, 26))
naive_calls = [count_calls_naive(n) for n in ns]
memo_calls = [max(2*n - 1, 1) for n in ns]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：线性坐标
ax1 = axes[0]
ax1.plot(ns, naive_calls, 'ro-', linewidth=2, markersize=5, label='朴素递归 O(2^n)')
ax1.plot(ns, memo_calls, 'bs-', linewidth=2, markersize=5, label='记忆化 O(n)')
ax1.set_xlabel('n', fontsize=12)
ax1.set_ylabel('函数调用次数', fontsize=12)
ax1.set_title('调用次数对比', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# 右图：对数坐标
ax2 = axes[1]
ax2.semilogy(ns, naive_calls, 'ro-', linewidth=2, markersize=5, label='朴素递归')
ax2.semilogy(ns, memo_calls, 'bs-', linewidth=2, markersize=5, label='记忆化')
ax2.set_xlabel('n', fontsize=12)
ax2.set_ylabel('调用次数 (对数坐标)', fontsize=12)
ax2.set_title('对数坐标下的对比', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'当n=25时：')
print(f'  朴素递归调用次数: {naive_calls[-1]:,}')
print(f'  记忆化调用次数: {memo_calls[-1]}')


---

## 练习题 (Exercises)

### 练习1：基础递归
1. 用递归实现计算一个数字各位数之和（如 1234 -> 1+2+3+4 = 10）
2. 用递归实现字符串反转（如 "hello" -> "olleh"）
3. 用递归判断一个字符串是否是回文

### 练习2：递归分析
1. 画出 `factorial(4)` 的完整调用栈
2. 画出 `fib(6)` 的递归树，标出哪些调用是重复的
3. 汉诺塔移动 5 个圆盘需要多少步？10个呢？

### 练习3：编程挑战

In [ ]:
# ============================================
# 练习3参考答案
# ============================================

# (a) 递归计算各位数之和
def digit_sum(n):
    '''递归计算各位数之和'''
    if n < 10:
        return n  # 基线条件：只有一位数
    return n % 10 + digit_sum(n // 10)  # 最后一位 + 剩余位数之和

# (b) 递归字符串反转
def reverse_string(s):
    '''递归反转字符串'''
    if len(s) <= 1:
        return s  # 基线条件
    return s[-1] + reverse_string(s[:-1])  # 最后一个字符 + 反转剩余部分

# (c) 递归判断回文
def is_palindrome(s):
    '''递归判断回文'''
    s = s.lower().replace(' ', '')  # 忽略大小写和空格
    if len(s) <= 1:
        return True  # 基线条件
    if s[0] != s[-1]:
        return False  # 首尾不同
    return is_palindrome(s[1:-1])  # 检查去掉首尾后的子串

# 测试
print('=== 各位数之和 ===')
for n in [1234, 9999, 12345]:
    print(f'  digit_sum({n}) = {digit_sum(n)}')

print('\n=== 字符串反转 ===')
for s in ['hello', 'Python', 'A Level']:
    print(f'  reverse("{s}") = "{reverse_string(s)}"')

print('\n=== 回文判断 ===')
for s in ['racecar', 'hello', 'A man a plan a canal Panama', 'level']:
    print(f'  is_palindrome("{s}") = {is_palindrome(s)}')

# (d) 递归二分查找
def binary_search_recursive(arr, target, low=0, high=None):
    '''递归版二分查找'''
    if high is None:
        high = len(arr) - 1
    if low > high:
        return -1  # 基线条件：没找到
    mid = (low + high) // 2
    if arr[mid] == target:
        return mid  # 基线条件：找到了
    elif arr[mid] < target:
        return binary_search_recursive(arr, target, mid + 1, high)
    else:
        return binary_search_recursive(arr, target, low, mid - 1)

print('\n=== 递归二分查找 ===')
arr = [1, 3, 5, 7, 9, 11, 13, 15]
for target in [7, 10, 1, 15]:
    idx = binary_search_recursive(arr, target)


---

## 本节小结 (Summary)

| 核心概念 | 要点 |
|----------|------|
| **递归定义** | 函数调用自身，必须有基线条件和递归条件 |
| **调用栈** | 每次递归调用创建新栈帧，返回时弹出 |
| **经典算法** | 阶乘O(n)、斐波那契O(2^n)->记忆化O(n)、汉诺塔O(2^n) |
| **递归 vs 迭代** | 递归更优雅但有栈开销，迭代更高效 |
| **记忆化** | 缓存结果避免重复计算，可将指数级降为线性 |
| **适用场景** | 树遍历、分治算法、回溯算法 |

### 考试要点 (Exam Tips)

- 能用递归实现基本算法（阶乘、斐波那契、二分查找）
- 能手动跟踪递归的调用过程
- 能分析递归的时间和空间复杂度
- 理解基线条件的重要性
- 能解释记忆化如何优化递归

---
*本章完！复习要点：大O表示法 -> 排序算法 -> 数据结构 -> 递归*